# Stream the chat endpoint over HTTP with `httpx`

Sends a real HTTP request to a **running** `chat-service` and streams the response token by token, the way a client would. Unlike `test_chat.ipynb` (which drives the app in-process via `TestClient`), this goes over the wire to `localhost:8000`.

Start the server first (in `backend/`):

```bash
uv run uvicorn chat_service.asgi:app --reload
```

Then run the cells below.

In [1]:
import httpx

BASE_URL = "http://localhost:8000"
CHAT_URL = f"{BASE_URL}/api/v1/chat"

# Sanity check the server is up.
try:
    ping = httpx.get(f"{BASE_URL}/ping", timeout=5.0)
    print(f"GET /ping -> {ping.status_code} {ping.json()}")
except Exception as e:
    print(f"Server not reachable at {BASE_URL}: {type(e).__name__}: {e}")
    print("Start it with:  uv run uvicorn chat_service.asgi:app --reload")

GET /ping -> 200 {'message': 'pong'}


## Stream a single request

`httpx.stream` keeps the connection open and yields the body as it arrives. We print each chunk as it comes in and capture the `X-Session-Id` header the server returns.

In [2]:
session_id = None
with httpx.stream(
    "POST",
    CHAT_URL,
    json={"question": "Please write a limerick about Stockholm"},
    timeout=60.0,
) as resp:
    resp.raise_for_status()
    session_id = resp.headers.get("X-Session-Id")
    print(f"status: {resp.status_code}  |  X-Session-Id: {session_id}")
    print("--- streamed answer ---")
    for chunk in resp.iter_text():
        print(chunk, end="", flush=True)
print()

status: 200  |  X-Session-Id: 8601628a-fd1b-4fa3-87d3-867323964477
--- streamed answer ---
There once was a city, Stockholm,  
Where islands and waters all welcome.  
With cobblestones bright,  
And midsummer light,  
It charms every heart that has come.


## Continue the same session

Echo the `X-Session-Id` back as `session_id` to continue the conversation. (History isn't persisted yet, so the token only round-trips for now — this shows the client-side pattern.)

In [3]:
with httpx.stream(
    "POST",
    CHAT_URL,
    json={"question": "Now write the same limerick about another city", "session_id": session_id},
    timeout=60.0,
) as resp:
    resp.raise_for_status()
    print(f"echoed X-Session-Id matches: {resp.headers.get('X-Session-Id') == session_id}")
    print("--- streamed answer ---")
    for chunk in resp.iter_text():
        print(chunk, end="", flush=True)
print()

echoed X-Session-Id matches: True
--- streamed answer ---
Of course — here’s a new limerick about another city:

There once was a city named Rome,  
With ruins and streets built of stone.  
The fountains would gleam,  
Like scenes from a dream,  
And tourists would never stay home.

If you want, I can make it:
- about a specific city
- funnier
- more elegant
- in the same style as the original


## Async variant

The same request with `httpx.AsyncClient` + `aiter_text`, for use inside async code. Jupyter supports top-level `await`.

In [4]:
async with httpx.AsyncClient(timeout=60.0) as client:
    async with client.stream(
        "POST",
        CHAT_URL,
        json={"question": "Say hello in exactly three words."},
    ) as resp:
        resp.raise_for_status()
        print(f"X-Session-Id: {resp.headers.get('X-Session-Id')}")
        print("--- streamed answer ---")
        async for chunk in resp.aiter_text():
            print(chunk, end="", flush=True)
print()

X-Session-Id: e2bf69d2-2758-43e7-b873-ae3d0a88d49f
--- streamed answer ---
Hello to you
